# Sinhala QA Test Split Inference

Loads the merged Hugging Face model from one repo and evaluates `splits/test.jsonl`.

In [ ]:
%pip install -q torch transformers accelerate huggingface_hub hf_transfer safetensors

In [ ]:
import json
import os
import re
from pathlib import Path
from collections import Counter

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# Use this only if the merged repo is private.
# Requires HF_TOKEN in the environment.
login(token=os.environ["HF_TOKEN"])

In [ ]:
MODEL_ID = "isji/sinllama-1b-qa-v2-merged"

# Works in Modal whether you upload test.jsonl to /tmp or /tmp/splits.
TEST_FILE_CANDIDATES = [
    Path("/tmp/test.jsonl")
]
TEST_FILE = next((path for path in TEST_FILE_CANDIDATES if path.exists()), TEST_FILE_CANDIDATES[0])
OUTPUT_FILE = Path("test_predictions.jsonl")
MAX_NEW_TOKENS = 80

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."

print("Using test file:", TEST_FILE.resolve())
print("Exists:", TEST_FILE.exists())

In [ ]:
print(f"Loading merged model: {MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print("Merged QA model loaded.")

In [ ]:
def build_qa_prompt(context, question):
    return (
        "උපදෙස්: ඔබ සිංහල ප්‍රශ්න-පිළිතුරු සහායකයෙකි. පහත සපයා ඇති තොරතුරු (Context) පමණක් භාවිතා කරමින් "
        "ප්‍රශ්නයට නිවැරදි, සෘජු, කෙටි පිළිතුරක් දෙන්න. Context තුළ පිළිතුර නොමැති නම් හෝ ප්‍රමාණවත් තොරතුරු නොමැති නම්, "
        f"“{NO_ANSWER}” යනුවෙන් පමණක් පිළිතුරු දෙන්න. Context වලින් පිටත දැනුම, අනුමාන, හෝ අමතර විස්තර භාවිතා නොකරන්න.\n\n"
        f"තොරතුරු (Context): {context}\n"
        f"ප්‍රශ්නය: {question}\n"
        "පිළිතුර: ["
    )

def run_qa(context, question, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_qa_prompt(context, question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_text.split("පිළිතුර: [")[-1].strip()
    if "]" in answer:
        answer = answer.split("]", 1)[0].strip()
    return answer

print("Inference helper ready.")

In [ ]:
def normalize_answer(text):
    text = (text or "").strip()
    text = text.replace("]", "").replace("[", "")
    text = re.sub(r"\s+", " ", text)
    for suffix in (" ය.", "යි.", " ය", "යි"):
        if text.endswith(suffix):
            text = text[: -len(suffix)].strip()
    return text

def keyword_match(prediction, gold):
    pred = normalize_answer(prediction)
    target = normalize_answer(gold)
    if not pred or not target:
        return False
    if pred == target:
        return True
    if target in pred or pred in target:
        return True

    target_tokens = [token for token in target.split() if len(token) > 1]
    if not target_tokens:
        return False
    matched = sum(1 for token in target_tokens if token in pred)
    return matched / len(target_tokens) >= 0.6

print("Scoring helpers ready.")

In [ ]:
test_rows = []
with TEST_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_rows.append(json.loads(line))

print(f"Loaded test rows: {len(test_rows)}")
print(Counter(row["answerable"] for row in test_rows))

In [ ]:
# Optional quick smoke test before running all 194 rows.
row = test_rows[0]
pred = run_qa(row["context"], row["question"])

print("Question:", row["question"])
print("Gold:", NO_ANSWER if row["answerable"] is False else row["answer"])
print("Prediction:", pred)

In [ ]:
predictions = []
exact = 0
soft = 0
no_answer_correct = 0
unanswerable_total = 0

for idx, row in enumerate(test_rows, start=1):
    gold = NO_ANSWER if row["answerable"] is False else row["answer"]
    pred = run_qa(row["context"], row["question"])

    exact_ok = normalize_answer(pred) == normalize_answer(gold)
    soft_ok = keyword_match(pred, gold)
    no_answer_ok = row["answerable"] is False and normalize_answer(pred) == normalize_answer(NO_ANSWER)

    exact += int(exact_ok)
    soft += int(soft_ok)
    if row["answerable"] is False:
        unanswerable_total += 1
        no_answer_correct += int(no_answer_ok)

    predictions.append({
        "index": idx,
        "grade": row.get("grade"),
        "chapter": row.get("chapter"),
        "answerable": row.get("answerable"),
        "context": row.get("context"),
        "question": row.get("question"),
        "gold_answer": gold,
        "prediction": pred,
        "exact_match": exact_ok,
        "soft_match": soft_ok,
    })

    if idx % 10 == 0 or idx == len(test_rows):
        print(f"Done {idx}/{len(test_rows)}")

print("Finished inference.")

In [ ]:
total = len(predictions)
print("=== Test Results ===")
print(f"Total: {total}")
print(f"Exact match: {exact}/{total} ({(exact / max(total, 1)) * 100:.1f}%)")
print(f"Soft match: {soft}/{total} ({(soft / max(total, 1)) * 100:.1f}%)")
print(
    f"No-answer exact: {no_answer_correct}/{unanswerable_total} "
    f"({(no_answer_correct / max(unanswerable_total, 1)) * 100:.1f}%)"
)

In [ ]:
with OUTPUT_FILE.open("w", encoding="utf-8", newline="\n") as f:
    for item in predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved predictions to {OUTPUT_FILE}")

In [ ]:
# Inspect failed examples.
failed = [item for item in predictions if not item["soft_match"]]
print(f"Soft-match failures: {len(failed)}")

for item in failed[:10]:
    print("-" * 80)
    print("Q:", item["question"])
    print("Gold:", item["gold_answer"])
    print("Pred:", item["prediction"])

In [ ]:
# Print generated and expected answers for all test QAs.
for item in predictions:
    print("-" * 100)
    print(f"Index     : {item['index']}")
    print(f"Grade     : {item.get('grade')} | Chapter: {item.get('chapter')}")
    print(f"Answerable: {item.get('answerable')}")
    print(f"Question  : {item['question']}")
    print(f"Expected  : {item['gold_answer']}")
    print(f"Generated : {item['prediction']}")
    print(f"Exact     : {item['exact_match']} | Soft: {item['soft_match']}")